In [101]:
import pandas as pd
from sklearn.metrics import (accuracy_score,f1_score,classification_report)
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
import os
from dotenv import load_dotenv
# from google import genai
from groq import Groq
from collections import Counter


In [60]:
golden_df = pd.read_csv(
    "../data/golden/amazon_golden_set.csv"
)

In [61]:
eval_df = golden_df[
    golden_df["human_intent"].notna() &
    (golden_df["human_intent"] != "")
].copy()

print("Candidate pool:", len(golden_df))
print("Evaluation set:", len(eval_df))

Candidate pool: 213
Evaluation set: 213


In [62]:
y_true = golden_df["human_intent"]

# Majority-class prediction
majority_class = y_true.value_counts().idxmax()

y_pred_majority = [majority_class] * len(y_true)

print("Majority class:", majority_class)
print("Accuracy:", accuracy_score(y_true, y_pred_majority))
print("Macro F1:", f1_score(y_true, y_pred_majority, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred_majority, average="weighted"))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred_majority,
        zero_division=0
    )
)

Majority class: OTHER
Accuracy: 0.17370892018779344
Macro F1: 0.0185
Weighted F1: 0.05141784037558685

Classification Report:
                       precision    recall  f1-score   support

       ACCOUNT_ACCESS       0.00      0.00      0.00        15
     ACCOUNT_SECURITY       0.00      0.00      0.00        12
             CASHBACK       0.00      0.00      0.00         8
       DELIVERY_DELAY       0.00      0.00      0.00        11
DELIVERY_DRIVER_ISSUE       0.00      0.00      0.00         8
DELIVERY_NOT_RECEIVED       0.00      0.00      0.00        16
    DELIVERY_TRACKING       0.00      0.00      0.00        16
         MISSING_ITEM       0.00      0.00      0.00         6
   ORDER_CANCELLATION       0.00      0.00      0.00        19
         ORDER_STATUS       0.00      0.00      0.00        12
                OTHER       0.17      1.00      0.30        37
        PRODUCT_ISSUE       0.00      0.00      0.00        13
        REFUND_STATUS       0.00      0.00      0.00  

In [63]:
X = golden_df["customer_text"].fillna("")
y = golden_df["human_intent"]

tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

In [64]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

y_pred_tfidf = cross_val_predict(
    tfidf_lr,
    X,
    y,
    cv=skf
)

In [65]:
tfidf_accuracy = accuracy_score(y, y_pred_tfidf)
tfidf_macro_f1 = f1_score(y, y_pred_tfidf, average="macro")
tfidf_weighted_f1 = f1_score(y, y_pred_tfidf, average="weighted")

print("TF-IDF + Logistic Regression")
print("--------------------------------")
print("Accuracy:", tfidf_accuracy)
print("Macro F1:", tfidf_macro_f1)
print("Weighted F1:", tfidf_weighted_f1)

print("\nClassification Report:")
print(
    classification_report(
        y,
        y_pred_tfidf,
        zero_division=0
    )
)

TF-IDF + Logistic Regression
--------------------------------
Accuracy: 0.352112676056338
Macro F1: 0.33439024786554683
Weighted F1: 0.34370646215830913

Classification Report:
                       precision    recall  f1-score   support

       ACCOUNT_ACCESS       0.53      0.53      0.53        15
     ACCOUNT_SECURITY       0.43      0.50      0.46        12
             CASHBACK       0.46      0.75      0.57         8
       DELIVERY_DELAY       0.00      0.00      0.00        11
DELIVERY_DRIVER_ISSUE       0.17      0.25      0.20         8
DELIVERY_NOT_RECEIVED       0.38      0.31      0.34        16
    DELIVERY_TRACKING       0.59      0.62      0.61        16
         MISSING_ITEM       0.00      0.00      0.00         6
   ORDER_CANCELLATION       0.47      0.47      0.47        19
         ORDER_STATUS       0.24      0.33      0.28        12
                OTHER       0.46      0.16      0.24        37
        PRODUCT_ISSUE       0.25      0.31      0.28        13
   

In [66]:
baseline_results = pd.DataFrame({
    "Model": [
        "Majority Class",
        "TF-IDF + Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_true, y_pred_majority),
        tfidf_accuracy
    ],
    "Macro F1": [
        f1_score(y_true, y_pred_majority, average="macro"),
        tfidf_macro_f1
    ],
    "Weighted F1": [
        f1_score(y_true, y_pred_majority, average="weighted"),
        tfidf_weighted_f1
    ]
})

baseline_results

,Model,Accuracy,Macro F1,Weighted F1
0,Majority Class,0.173709,0.01850,0.051418
1,TF-IDF + Logistic Regression,0.352113,0.33439,0.343706


## Semantic Intent Classiffier

In [67]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model_multi = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2421.67it/s]


In [68]:
processed_path = Path("../data/processed")

amazon_pairs = pd.read_parquet(
    processed_path / "amazon_support_pairs.parquet"
)

print("Historical pairs:", len(amazon_pairs))

Historical pairs: 166963


In [69]:
intent_examples = {
    "DELIVERY_DELAY": [
        "My package is late",
        "My order was supposed to arrive yesterday",
        "The promised delivery date has passed",
        "Why is my delivery delayed?"
    ],

    "DELIVERY_NOT_RECEIVED": [
        "My package hasn't arrived",
        "I still haven't received my package",
        "My order never arrived",
        "I have not received my delivery"
    ],

    "DELIVERY_TRACKING": [
        "Where can I track my order?",
        "My tracking information has not updated",
        "The tracking status is stuck",
        "I want to know where my package is"
    ],

    "DELIVERY_DRIVER_ISSUE": [
        "The delivery driver did not follow my instructions",
        "The driver could not find my address",
        "I had a problem with the delivery driver",
        "The delivery driver marked an incorrect delivery attempt"
    ],

    "ORDER_STATUS": [
        "What is the status of my order?",
        "Can you tell me about my order?",
        "I want to know the current status of my order",
        "What is happening with my order?"
    ],

    "ORDER_CANCELLATION": [
        "I want to cancel my order",
        "Please cancel my order",
        "Why was my order cancelled?",
        "I need to cancel an order"
    ],

    "MISSING_ITEM": [
        "One item from my order is missing",
        "I ordered two items but only received one",
        "Part of my order is missing",
        "An item was missing from my package"
    ],

    "WRONG_ITEM_RECEIVED": [
        "I received the wrong item",
        "The product I received is not what I ordered",
        "I got a different product than I ordered",
        "You sent me the wrong product"
    ],

    "RETURN_REPLACEMENT": [
        "I want to return this item",
        "I need to replace this product",
        "How can I return my order?",
        "I want a replacement"
    ],

    "REFUND_STATUS": [
        "Where is my refund?",
        "My refund has not arrived",
        "When will I receive my refund?",
        "I am still waiting for my refund"
    ],

    "ACCOUNT_ACCESS": [
        "I cannot log into my account",
        "I forgot my password",
        "I am locked out of my account",
        "I cannot access my Amazon account"
    ],

    "ACCOUNT_SECURITY": [
        "Someone hacked my account",
        "I think someone accessed my account",
        "There is suspicious activity on my account",
        "My account has been compromised"
    ],

    "UNEXPECTED_CHARGE": [
        "I was charged unexpectedly",
        "Why was I charged this amount?",
        "I don't recognize this charge",
        "I was charged for something I did not purchase"
    ],

    "CASHBACK": [
        "I have not received my cashback",
        "Where is my cashback?",
        "My cashback is missing",
        "I was supposed to get cashback"
    ],

    "PRODUCT_ISSUE": [
        "The product is damaged",
        "The item is defective",
        "My product is not working",
        "The product I received is broken"
    ],

    "OTHER": [
        "I have a question",
        "I need help",
        "Thank you for your help",
        "This is not related to the supported issues"
    ]
}

In [70]:
intent_names = list(intent_examples.keys())

prototype_texts = []
prototype_intents = []

for intent, examples in intent_examples.items():
    for example in examples:
        prototype_texts.append(example)
        prototype_intents.append(intent)

prototype_embeddings = model_multi.encode(
    prototype_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Prototype examples:", len(prototype_texts))
print("Embedding shape:", prototype_embeddings.shape)

Batches: 100%|██████████| 2/2 [00:00<00:00, 10.95it/s]

Prototype examples: 64
Embedding shape: (64, 384)


In [71]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def predict_intent(text):
    """
    Predict intent using semantic similarity to intent prototypes.
    Returns:
        intent
        confidence
        margin
    """
    
    embedding = model_multi.encode(
        [str(text)],
        normalize_embeddings=True
    )
    
    similarities = cosine_similarity(
        embedding,
        prototype_embeddings
    )[0]
    
    # Best prototype
    best_idx = np.argmax(similarities)
    best_intent = prototype_intents[best_idx]
    best_score = similarities[best_idx]
    
    # Second-best score
    sorted_scores = np.sort(similarities)[::-1]
    second_score = sorted_scores[1]
    
    margin = best_score - second_score
    
    return best_intent, best_score, margin

In [72]:
test_messages = [
    "My package was supposed to arrive yesterday",
    "I received the wrong product",
    "Someone hacked my Amazon account",
    "Where is my refund?",
    "I ordered two items but only received one"
]

for message in test_messages:
    intent, score, margin = predict_intent(message)
    
    print(f"Message: {message}")
    print(f"Intent: {intent}")
    print(f"Similarity: {score:.3f}")
    print(f"Margin: {margin:.3f}")
    print("-" * 60)

Message: My package was supposed to arrive yesterday
Intent: DELIVERY_DELAY
Similarity: 0.792
Margin: 0.037
------------------------------------------------------------
Message: I received the wrong product
Intent: WRONG_ITEM_RECEIVED
Similarity: 0.836
Margin: 0.004
------------------------------------------------------------
Message: Someone hacked my Amazon account
Intent: ACCOUNT_SECURITY
Similarity: 0.808
Margin: 0.067
------------------------------------------------------------
Message: Where is my refund?
Intent: REFUND_STATUS
Similarity: 1.000
Margin: 0.121
------------------------------------------------------------
Message: I ordered two items but only received one
Intent: MISSING_ITEM
Similarity: 1.000
Margin: 0.238
------------------------------------------------------------


In [73]:
golden_predictions = []

for text in golden_df["customer_text"]:
    intent, score, margin = predict_intent(text)
    
    golden_predictions.append({
        "predicted_intent": intent,
        "similarity": score,
        "margin": margin
    })

semantic_results = pd.DataFrame(golden_predictions)

semantic_results.head()

,predicted_intent,similarity,margin
0,ACCOUNT_SECURITY,0.747296,0.068258
1,DELIVERY_NOT_RECEIVED,0.733459,0.039342
2,DELIVERY_NOT_RECEIVED,0.709770,0.082033
3,PRODUCT_ISSUE,0.665636,0.023541
4,REFUND_STATUS,0.606183,0.000336


In [74]:
semantic_eval = pd.concat(
    [
        golden_df.reset_index(drop=True),
        semantic_results
    ],
    axis=1
)

semantic_eval.head()

,customer_tweet_id,customer_text,customer_text.1,human_intent,difficulty,notes,predicted_intent,similarity,margin
0,275129,@115821 Please help! Someone hacked my account and I can’t contact you on your website as they changed the email and I cannot log in!,@115821 Please help! Someone hacked my account and I can’t contact you on your website as they changed the email and I cannot log in!,ACCOUNT_SECURITY,EASY,Explicitly hacked; email changed,ACCOUNT_SECURITY,0.747296,0.068258
1,2135765,"The @116090 app shows that my package was delivered yesterday, yet here I am without said package.","The @116090 app shows that my package was delivered yesterday, yet here I am without said package.",DELIVERY_NOT_RECEIVED,EASY,Delivered notification but package not received,DELIVERY_NOT_RECEIVED,0.733459,0.039342
2,1458000,My order from @115821 just was delivered. Opened it and there’s NOTHING IN IT,My order from @115821 just was delivered. Opened it and there’s NOTHING IN IT,MISSING_ITEM,EASY,Order arrived but contents were missing,DELIVERY_NOT_RECEIVED,0.709770,0.082033
3,2513448,@115850 A damaged product was supposed to be collected a few days back. But instead the person hasn't collected it yet. Very disappointed.,@115850 A damaged product was supposed to be collected a few days back. But instead the person hasn't collected it yet. Very disappointed.,PRODUCT_ISSUE,BOUNDARY,Damaged product; collection issue is secondary,PRODUCT_ISSUE,0.665636,0.023541
4,1144950,@AmazonHelp Thanks for the super quick reply! Will this work even if I just want a partial refund?,@AmazonHelp Thanks for the super quick reply! Will this work even if I just want a partial refund?,REFUND_STATUS,BOUNDARY,Asking about partial refund,REFUND_STATUS,0.606183,0.000336


In [75]:
semantic_accuracy = accuracy_score(
    semantic_eval["human_intent"],
    semantic_eval["predicted_intent"]
)

semantic_macro_f1 = f1_score(
    semantic_eval["human_intent"],
    semantic_eval["predicted_intent"],
    average="macro"
)

semantic_weighted_f1 = f1_score(
    semantic_eval["human_intent"],
    semantic_eval["predicted_intent"],
    average="weighted"
)

print("Semantic Prototype Classifier")
print("--------------------------------")
print("Accuracy:", semantic_accuracy)
print("Macro F1:", semantic_macro_f1)
print("Weighted F1:", semantic_weighted_f1)

print("\nClassification Report:")
print(
    classification_report(
        semantic_eval["human_intent"],
        semantic_eval["predicted_intent"],
        zero_division=0
    )
)

Semantic Prototype Classifier
--------------------------------
Accuracy: 0.4272300469483568
Macro F1: 0.4305882867708175
Weighted F1: 0.40025407811776353

Classification Report:
                       precision    recall  f1-score   support

       ACCOUNT_ACCESS       0.22      0.67      0.33        15
     ACCOUNT_SECURITY       0.50      0.42      0.45        12
             CASHBACK       0.50      0.50      0.50         8
       DELIVERY_DELAY       0.20      0.27      0.23        11
DELIVERY_DRIVER_ISSUE       0.70      0.88      0.78         8
DELIVERY_NOT_RECEIVED       0.37      0.44      0.40        16
    DELIVERY_TRACKING       0.64      0.56      0.60        16
         MISSING_ITEM       0.25      0.33      0.29         6
   ORDER_CANCELLATION       0.68      0.68      0.68        19
         ORDER_STATUS       0.55      0.50      0.52        12
                OTHER       0.00      0.00      0.00        37
        PRODUCT_ISSUE       0.71      0.38      0.50        13
  

In [76]:
embedding_path = Path("../data/processed/amazon_customer_embeddings.npy")

print(embedding_path)

..\data\processed\amazon_customer_embeddings.npy


In [77]:
if embedding_path.exists():
    historical_embeddings = np.load(embedding_path)
    print("Loaded embeddings:", historical_embeddings.shape)

else:
    historical_embeddings = model_multi.encode(
        amazon_pairs["customer_text_clean"].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    np.save(
        embedding_path,
        historical_embeddings
    )

    print("Saved embeddings:", historical_embeddings.shape)

Loaded embeddings: (166963, 384)


In [78]:
def retrieve_similar_cases(
    query,
    top_k=5,
    candidate_k=50
):
    query_embedding = model_multi.encode(
        [query],
        normalize_embeddings=True
    )[0]

    scores = historical_embeddings @ query_embedding

    candidate_indices = np.argsort(scores)[::-1][:candidate_k]

    results = amazon_pairs.iloc[candidate_indices].copy()
    results["similarity"] = scores[candidate_indices]

    return results.head(top_k)

In [79]:
query = "My package was supposed to arrive yesterday but it still hasn't arrived."

results = retrieve_similar_cases(query, top_k=5)

results[
    [
        "customer_text_clean",
        "support_text",
        "similarity"
    ]
]

,customer_text_clean,support_text,similarity
38557,My package was supposed to be delivered today. It has still not arrived,@235783 I'm sorry your packages hasn't arrived yet. What does your current tracking information say? You can check that here: https://t.co/aaDyEz1VgE ^EA,0.950095
108232,my package was supposed to arrive yesterday and it didn’t sooooooo,@512424 I'm sorry for the wait! Will you tell us what the most up to date tracking says here: https://t.co/Y5jpI9gRhE? ^DW,0.920710
94652,"said my package was delivered yesterday, yet it still isn’t here 😐",@165088 Well there could be a couple of places it could be hiding. Click here to begin the search: https://t.co/jEGovQGVPU ^DA,0.916020
61775,Well my package was supposed to be delivered today but it hasn't.,@334206 I'm sorry you haven't received your order! Have you been notified by any delays via e-mail? Please check and let us know! ^KJ,0.911325
99993,Somehow and didn't get my package delivered today that said it was going to be,@475923 I'm sorry for the delay! We want to help if we can. Please keep us posted on the outcome with the carrier. ^MO,0.897767


In [80]:
pd.set_option("display.max_colwidth", 300)

results[
    [
        "customer_text_clean",
        "support_text",
        "similarity"
    ]
].to_string(index=False)

"                                                           customer_text_clean                                                                                                                                                support_text  similarity\n       My package was supposed to be delivered today. It has still not arrived @235783 I'm sorry your packages hasn't arrived yet. What does your current tracking information say? You can check that here:  https://t.co/aaDyEz1VgE  ^EA    0.950095\n            my package was supposed to arrive yesterday and it didn’t sooooooo                                  @512424 I'm sorry for the wait! Will you tell us what the most up to date tracking says here: https://t.co/Y5jpI9gRhE? ^DW    0.920710\n            said my package was delivered yesterday, yet it still isn’t here 😐                              @165088 Well there could be a couple of places it could be hiding. Click here to begin the search: https://t.co/jEGovQGVPU ^DA    0.916020\n       

In [81]:
prototype_similarity = historical_embeddings @ prototype_embeddings.T

best_prototype_indices = np.argmax(
    prototype_similarity,
    axis=1
)

amazon_pairs["predicted_intent"] = [
    prototype_intents[i]
    for i in best_prototype_indices
]

amazon_pairs["intent_similarity"] = prototype_similarity[
    np.arange(len(amazon_pairs)),
    best_prototype_indices
]

print(amazon_pairs[
    ["customer_text_clean", "predicted_intent", "intent_similarity"]
].head())

                                             customer_text_clean  \
0                                       amazonのfireTVstickが見れない😢   
1            電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんでしょうね。   
2                                              こちらこそありがとうございました。   
3                                       amazonプライムビデオ、再生エラーが多いです   
4  Way to drop the ball on customer service so pissed right now!   

      predicted_intent  intent_similarity  
0       ACCOUNT_ACCESS           0.407557  
1  WRONG_ITEM_RECEIVED           0.482171  
2                OTHER           0.628228  
3       ACCOUNT_ACCESS           0.431274  
4   ORDER_CANCELLATION           0.448575  


In [82]:
silver_path = Path("../data/processed/amazon_support_pairs_silver.parquet")

amazon_pairs.to_parquet(
    silver_path,
    index=False
)

print("Saved:", silver_path)
print("Rows:", len(amazon_pairs))

Saved: ..\data\processed\amazon_support_pairs_silver.parquet
Rows: 166963


In [83]:
amazon_pairs["predicted_intent"].value_counts()

predicted_intent
DELIVERY_DELAY           30283
ACCOUNT_ACCESS           21001
WRONG_ITEM_RECEIVED      18670
DELIVERY_NOT_RECEIVED    13962
OTHER                    11746
REFUND_STATUS            10602
ORDER_CANCELLATION        9419
ACCOUNT_SECURITY          9085
ORDER_STATUS              7649
MISSING_ITEM              6827
DELIVERY_DRIVER_ISSUE     6434
DELIVERY_TRACKING         5154
PRODUCT_ISSUE             4621
CASHBACK                  4485
RETURN_REPLACEMENT        3853
UNEXPECTED_CHARGE         3172
Name: count, dtype: int64

In [84]:
query = "My package was supposed to arrive yesterday but it still hasn't arrived."

results = retrieve_similar_cases(
    query,
    top_k=5
)

results[
    [
        "customer_text_clean",
        "support_text",
        "similarity"
    ]
]

,customer_text_clean,support_text,similarity
38557,My package was supposed to be delivered today. It has still not arrived,@235783 I'm sorry your packages hasn't arrived yet. What does your current tracking information say? You can check that here: https://t.co/aaDyEz1VgE ^EA,0.950095
108232,my package was supposed to arrive yesterday and it didn’t sooooooo,@512424 I'm sorry for the wait! Will you tell us what the most up to date tracking says here: https://t.co/Y5jpI9gRhE? ^DW,0.920710
94652,"said my package was delivered yesterday, yet it still isn’t here 😐",@165088 Well there could be a couple of places it could be hiding. Click here to begin the search: https://t.co/jEGovQGVPU ^DA,0.916020
61775,Well my package was supposed to be delivered today but it hasn't.,@334206 I'm sorry you haven't received your order! Have you been notified by any delays via e-mail? Please check and let us know! ^KJ,0.911325
99993,Somehow and didn't get my package delivered today that said it was going to be,@475923 I'm sorry for the delay! We want to help if we can. Please keep us posted on the outcome with the carrier. ^MO,0.897767


## Grounded Response Generation

In [85]:
def build_evidence_context(results):
    evidence = []

    for i, row in results.iterrows():
        evidence.append(
            f"""Historical Case {len(evidence) + 1}
Customer issue:
{row['customer_text_clean']}

AmazonHelp response:
{row['support_text']}

Similarity:
{row['similarity']:.3f}
"""
        )

    return "\n---\n".join(evidence)

In [86]:
query = "My package was supposed to arrive yesterday but it still hasn't arrived."

results = retrieve_similar_cases(
    query,
    top_k=5
)

context = build_evidence_context(results)

print(context)

Historical Case 1
Customer issue:
My package was supposed to be delivered today. It has still not arrived

AmazonHelp response:
@235783 I'm sorry your packages hasn't arrived yet. What does your current tracking information say? You can check that here:  https://t.co/aaDyEz1VgE  ^EA

Similarity:
0.950

---
Historical Case 2
Customer issue:
my package was supposed to arrive yesterday and it didn’t sooooooo

AmazonHelp response:
@512424 I'm sorry for the wait! Will you tell us what the most up to date tracking says here: https://t.co/Y5jpI9gRhE? ^DW

Similarity:
0.921

---
Historical Case 3
Customer issue:
said my package was delivered yesterday, yet it still isn’t here 😐

AmazonHelp response:
@165088 Well there could be a couple of places it could be hiding. Click here to begin the search: https://t.co/jEGovQGVPU ^DA

Similarity:
0.916

---
Historical Case 4
Customer issue:
Well my package was supposed to be delivered today but it hasn't.

AmazonHelp response:
@334206 I'm sorry you have

In [87]:
def build_generation_prompt(query, results):
    context = build_evidence_context(results)

    prompt = f"""
You are an Amazon customer support assistant.

Your task is to draft a helpful response to the customer's issue
using the historical AmazonHelp responses below as evidence.

CUSTOMER ISSUE:
{query}

HISTORICAL AMAZONHELP CASES:
{context}

INSTRUCTIONS:
1. Ground your response in the historical support behavior shown above.
2. Do not invent policies, refunds, delivery guarantees, or actions.
3. Do not copy a historical response verbatim.
4. Adapt the response to the customer's specific situation.
5. If the historical cases suggest checking tracking information, ask the
   customer to check or provide the latest tracking status.
6. Keep the response concise and professional.
7. Do not include URLs unless they are present in the historical evidence.

Draft only the customer-facing response.
"""
    return prompt

In [88]:
query = "My package was supposed to arrive yesterday but it still hasn't arrived."

results = retrieve_similar_cases(query, top_k=5)

prompt = build_generation_prompt(query, results)

print("Prompt length:", len(prompt))
print(prompt[:3000])

Prompt length: 2261

You are an Amazon customer support assistant.

Your task is to draft a helpful response to the customer's issue
using the historical AmazonHelp responses below as evidence.

CUSTOMER ISSUE:
My package was supposed to arrive yesterday but it still hasn't arrived.

HISTORICAL AMAZONHELP CASES:
Historical Case 1
Customer issue:
My package was supposed to be delivered today. It has still not arrived

AmazonHelp response:
@235783 I'm sorry your packages hasn't arrived yet. What does your current tracking information say? You can check that here:  https://t.co/aaDyEz1VgE  ^EA

Similarity:
0.950

---
Historical Case 2
Customer issue:
my package was supposed to arrive yesterday and it didn’t sooooooo

AmazonHelp response:
@512424 I'm sorry for the wait! Will you tell us what the most up to date tracking says here: https://t.co/Y5jpI9gRhE? ^DW

Similarity:
0.921

---
Historical Case 3
Customer issue:
said my package was delivered yesterday, yet it still isn’t here 😐

Amazon

In [95]:
load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

@{customer_handle} I’m sorry you still haven’t received your order. Could you let us know what the most recent tracking status shows? You can view it here: https://t.co/aaDyEz1VgE. Also, please check your email for any carrier delay notices and keep us posted.


In [90]:
import time

def generate_response(query, results, max_retries=3):
    prompt = build_generation_prompt(query, results)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            return response.choices[0].message.content

        except Exception as e:
            error_message = str(e)

            if "429" in error_message or "rate" in error_message.lower():
                wait_time = 2 ** attempt

                print(
                    f"Groq temporarily rate-limited. "
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

            else:
                raise

    raise RuntimeError(
        "Groq API remained unavailable after retries."
    )

In [96]:
query = "You are a fool"

results = retrieve_similar_cases(query, top_k=5)

response = generate_response(query, results)

print(response)

We’re sorry you’ve had a frustrating experience. If you’re referring to a specific order or request, could you please share the order number or reply to the email you received from our Executive Customer Relations team? Once we have those details we’ll be able to look into it further and address your concerns.


## Decision Engine

In [98]:
retrieval_eval = []

for idx, row in golden_df.iterrows():

    results = retrieve_similar_cases(
        row["customer_text"],
        top_k=5
    )

    retrieval_eval.append({
        "golden_index": idx,
        "human_intent": row["human_intent"],
        "top_similarity": results.iloc[0]["similarity"],
        "mean_top5_similarity": results["similarity"].mean(),
        "min_top5_similarity": results["similarity"].min()
    })

retrieval_eval = pd.DataFrame(retrieval_eval)

retrieval_eval.describe()

,golden_index,top_similarity,mean_top5_similarity,min_top5_similarity
count,213.000000,213.000000,213.000000,213.000000
mean,106.000000,0.897140,0.831945,0.801228
std,61.631972,0.056827,0.049817,0.063478
min,0.000000,0.746671,0.664676,0.554371
25%,53.000000,0.852185,0.804886,0.763490
50%,106.000000,0.888035,0.834509,0.815441
75%,159.000000,0.949399,0.866099,0.844941
max,212.000000,1.000000,0.956634,0.932369


In [99]:
retrieval_eval[
    ["top_similarity",
     "mean_top5_similarity",
     "min_top5_similarity"]
].describe()

,top_similarity,mean_top5_similarity,min_top5_similarity
count,213.000000,213.000000,213.000000
mean,0.897140,0.831945,0.801228
std,0.056827,0.049817,0.063478
min,0.746671,0.664676,0.554371
25%,0.852185,0.804886,0.763490
50%,0.888035,0.834509,0.815441
75%,0.949399,0.866099,0.844941
max,1.000000,0.956634,0.932369


In [102]:

retrieval_eval = []

for idx, row in golden_df.iterrows():

    results = retrieve_similar_cases(
        row["customer_text"],
        top_k=5
    )

    retrieved_intents = results["predicted_intent"].tolist()

    intent_counts = Counter(retrieved_intents)

    most_common_intent, most_common_count = intent_counts.most_common(1)[0]

    agreement = most_common_count / len(retrieved_intents)

    retrieval_eval.append({
        "golden_index": idx,
        "human_intent": row["human_intent"],
        "top_similarity": results.iloc[0]["similarity"],
        "mean_top5_similarity": results["similarity"].mean(),
        "min_top5_similarity": results["similarity"].min(),
        "retrieved_intent": most_common_intent,
        "evidence_agreement": agreement
    })

retrieval_eval = pd.DataFrame(retrieval_eval)

In [103]:
retrieval_eval["evidence_agreement"].value_counts().sort_index()

evidence_agreement
0.2     1
0.4    25
0.6    54
0.8    53
1.0    80
Name: count, dtype: int64

In [104]:
retrieval_eval["intent_match"] = (
    retrieval_eval["retrieved_intent"]
    == retrieval_eval["human_intent"]
)

print(
    "Retrieved majority intent matches human intent:",
    retrieval_eval["intent_match"].mean()
)

print(
    retrieval_eval["intent_match"].value_counts()
)

Retrieved majority intent matches human intent: 0.3615023474178404
intent_match
False    136
True      77
Name: count, dtype: int64


In [105]:
pd.crosstab(
    retrieval_eval["evidence_agreement"],
    retrieval_eval["intent_match"],
    normalize="index"
)

intent_match,False,True
evidence_agreement,,
0.2,1.000000,0.000000
0.4,0.800000,0.200000
0.6,0.722222,0.277778
0.8,0.773585,0.226415
1.0,0.437500,0.562500


In [106]:
intent_eval = []

for idx, row in golden_df.iterrows():

    intent, similarity, margin = predict_intent(
        row["customer_text"]
    )

    intent_eval.append({
        "golden_index": idx,
        "human_intent": row["human_intent"],
        "predicted_intent": intent,
        "intent_similarity": similarity,
        "intent_margin": margin
    })

intent_eval = pd.DataFrame(intent_eval)

intent_eval.describe()

,golden_index,intent_similarity,intent_margin
count,213.000000,213.000000,213.000000
mean,106.000000,0.625619,0.051336
std,61.631972,0.115585,0.054617
min,0.000000,0.230968,0.000187
25%,53.000000,0.560403,0.010895
50%,106.000000,0.641203,0.039233
75%,159.000000,0.700216,0.068730
max,212.000000,0.873103,0.300193


In [107]:
intent_eval[
    ["intent_similarity", "intent_margin"]
].describe()

,intent_similarity,intent_margin
count,213.000000,213.000000
mean,0.625619,0.051336
std,0.115585,0.054617
min,0.230968,0.000187
25%,0.560403,0.010895
50%,0.641203,0.039233
75%,0.700216,0.068730
max,0.873103,0.300193


In [108]:
decision_eval = retrieval_eval.merge(
    intent_eval,
    on=["golden_index", "human_intent"],
    how="inner"
)

print(decision_eval.shape)

decision_eval.head()

(213, 11)


,golden_index,human_intent,top_similarity,mean_top5_similarity,min_top5_similarity,retrieved_intent,evidence_agreement,intent_match,predicted_intent,intent_similarity,intent_margin
0,0,ACCOUNT_SECURITY,0.978056,0.878322,0.846552,ACCOUNT_SECURITY,1.0,True,ACCOUNT_SECURITY,0.747296,0.068258
1,1,DELIVERY_NOT_RECEIVED,0.948658,0.844523,0.805203,DELIVERY_NOT_RECEIVED,1.0,True,DELIVERY_NOT_RECEIVED,0.733459,0.039342
2,2,MISSING_ITEM,0.912221,0.817612,0.787190,DELIVERY_NOT_RECEIVED,0.8,False,DELIVERY_NOT_RECEIVED,0.709770,0.082033
3,3,PRODUCT_ISSUE,0.953599,0.826182,0.739113,PRODUCT_ISSUE,1.0,True,PRODUCT_ISSUE,0.665636,0.023541
4,4,REFUND_STATUS,0.833455,0.829311,0.824349,REFUND_STATUS,1.0,True,REFUND_STATUS,0.606183,0.000336


In [109]:
decision_eval.groupby("human_intent")[
    [
        "intent_similarity",
        "intent_margin",
        "top_similarity",
        "mean_top5_similarity"
    ]
].agg(["mean", "median"])

intent_similarity           intent_margin            \
                                   mean    median          mean    median   
human_intent                                                                
ACCOUNT_ACCESS                 0.719247  0.764675      0.118554  0.057877   
ACCOUNT_SECURITY               0.676053  0.694986      0.038794  0.034969   
CASHBACK                       0.665544  0.674359      0.052365  0.054134   
DELIVERY_DELAY                 0.647884  0.704665      0.059404  0.045082   
DELIVERY_DRIVER_ISSUE          0.630518  0.658202      0.029474  0.014209   
DELIVERY_NOT_RECEIVED          0.667364  0.669632      0.030521  0.032471   
DELIVERY_TRACKING              0.604549  0.583843      0.063220  0.062441   
MISSING_ITEM                   0.635022  0.649650      0.032816  0.026924   
ORDER_CANCELLATION             0.672479  0.706304      0.025263  0.015028   
ORDER_STATUS                   0.655293  0.627784      0.028620  0.022495   
OTHER                          0.536099  0.551087      0.073176  0.052092   
PRODUCT_ISSUE                  0.580513  0.575311      0.032569  0.025481   
REFUND_STATUS                  0.642236  0.628551      0.022675  0.017268   
RETURN_REPLACEMENT             0.599454  0.567787      0.052502  0.051381   
UNEXPECTED_CHARGE              0.590806  0.585946      0.073206  0.076919   
WRONG_ITEM_RECEIVED            0.609238  0.602882      0.042919  0.048166   

                      top_similarity           mean_top5_similarity            
                                mean    median                 mean    median  
human_intent                                                                   
ACCOUNT_ACCESS              0.886795  0.880337             0.833307  0.844991  
ACCOUNT_SECURITY            0.918057  0.955121             0.843929  0.834625  
CASHBACK                    0.937232  0.951400             0.858704  0.866833  
DELIVERY_DELAY              0.906720  0.893883             0.843670  0.841268  
DELIVERY_DRIVER_ISSUE       0.920838  0.939605             0.824180  0.818066  
DELIVERY_NOT_RECEIVED       0.906131  0.897411             0.844707  0.845629  
DELIVERY_TRACKING           0.881245  0.873890             0.817060  0.818026  
MISSING_ITEM                0.852718  0.872915             0.823227  0.823574  
ORDER_CANCELLATION          0.904352  0.878379             0.860547  0.860704  
ORDER_STATUS                0.898446  0.902798             0.859060  0.853159  
OTHER                       0.891369  0.882807             0.811653  0.823668  
PRODUCT_ISSUE               0.920547  0.940697             0.800017  0.801276  
REFUND_STATUS               0.891718  0.883578             0.852760  0.848222  
RETURN_REPLACEMENT          0.868890  0.868511             0.828342  0.831294  
UNEXPECTED_CHARGE           0.873097  0.870736             0.792976  0.798691  
WRONG_ITEM_RECEIVED         0.894094  0.847865             0.825448  0.789576

## Escalation Decision Engine

In [110]:
RISK_INTENTS = {
    "ACCOUNT_SECURITY",
    "UNEXPECTED_CHARGE"
}

def make_decision(
    predicted_intent,
    intent_margin,
    top_similarity
):
    if predicted_intent == "OTHER":
        return "ESCALATE", "Message is outside the supported intent taxonomy."

    if predicted_intent in RISK_INTENTS:
        return "ESCALATE", f"{predicted_intent} is treated as high-risk and requires human review."

    if intent_margin < 0.03:
        return "ESCALATE", "Intent classification is ambiguous."

    if top_similarity < 0.85:
        return "ESCALATE", "Historical retrieval evidence is weak."

    return "AUTO_HANDLE", "Intent is sufficiently confident and relevant historical evidence was retrieved."

In [111]:
decisions = []

for _, row in decision_eval.iterrows():

    decision, reason = make_decision(
        predicted_intent=row["predicted_intent"],
        intent_margin=row["intent_margin"],
        top_similarity=row["top_similarity"]
    )

    decisions.append({
        "decision": decision,
        "decision_reason": reason
    })

decisions_df = pd.DataFrame(decisions)

decision_eval = pd.concat(
    [decision_eval.reset_index(drop=True),
     decisions_df],
    axis=1
)

In [112]:
decision_eval["decision"].value_counts()

decision
ESCALATE       129
AUTO_HANDLE     84
Name: count, dtype: int64

In [113]:
pd.crosstab(
    decision_eval["human_intent"],
    decision_eval["decision"],
    margins=True
)

decision,AUTO_HANDLE,ESCALATE,All
human_intent,,,
ACCOUNT_ACCESS,11,4,15
ACCOUNT_SECURITY,2,10,12
CASHBACK,7,1,8
DELIVERY_DELAY,6,5,11
DELIVERY_DRIVER_ISSUE,3,5,8
DELIVERY_NOT_RECEIVED,6,10,16
DELIVERY_TRACKING,8,8,16
MISSING_ITEM,2,4,6
ORDER_CANCELLATION,4,15,19


In [114]:
auto_handled = decision_eval[
    decision_eval["decision"] == "AUTO_HANDLE"
]

safe_automation_precision = (
    auto_handled["intent_match"].mean()
)

print(
    "AUTO-HANDLE cases:",
    len(auto_handled)
)

print(
    "Safe automation precision:",
    safe_automation_precision
)

AUTO-HANDLE cases: 84
Safe automation precision: 0.39285714285714285


In [115]:
automation_rate = (
    decision_eval["decision"] == "AUTO_HANDLE"
).mean()

print(
    "Automation rate:",
    automation_rate
)

Automation rate: 0.39436619718309857


In [118]:
auto_failures = decision_eval[
    (decision_eval["decision"] == "AUTO_HANDLE") &
    (~decision_eval["intent_match"])
].copy()

print("Incorrect AUTO-HANDLE cases:", len(auto_failures))

auto_failures[
    [
        "golden_index",
        "human_intent",
        "predicted_intent",
        "intent_similarity",
        "intent_margin",
        "top_similarity",
        "decision_reason"
    ]
]

Incorrect AUTO-HANDLE cases: 51


,golden_index,human_intent,predicted_intent,intent_similarity,intent_margin,top_similarity,decision_reason
2,2,MISSING_ITEM,DELIVERY_NOT_RECEIVED,0.709770,0.082033,0.912221,Intent is sufficiently confident and relevant historical evidence was retrieved.
6,6,ACCOUNT_SECURITY,ACCOUNT_ACCESS,0.487351,0.047775,0.983677,Intent is sufficiently confident and relevant historical evidence was retrieved.
10,10,ORDER_STATUS,DELIVERY_NOT_RECEIVED,0.784961,0.071071,0.902649,Intent is sufficiently confident and relevant historical evidence was retrieved.
11,11,DELIVERY_NOT_RECEIVED,WRONG_ITEM_RECEIVED,0.700216,0.067279,0.924554,Intent is sufficiently confident and relevant historical evidence was retrieved.
20,20,OTHER,ACCOUNT_ACCESS,0.320509,0.097600,1.000000,Intent is sufficiently confident and relevant historical evidence was retrieved.
24,24,OTHER,WRONG_ITEM_RECEIVED,0.496950,0.113363,0.895404,Intent is sufficiently confident and relevant historical evidence was retrieved.
28,28,DELIVERY_DELAY,DELIVERY_DELAY,0.713479,0.066497,0.904613,Intent is sufficiently confident and relevant historical evidence was retrieved.
32,32,DELIVERY_DELAY,DELIVERY_NOT_RECEIVED,0.712666,0.104748,0.964124,Intent is sufficiently confident and relevant historical evidence was retrieved.
37,37,DELIVERY_DELAY,DELIVERY_NOT_RECEIVED,0.756594,0.034059,0.893883,Intent is sufficiently confident and relevant historical evidence was retrieved.
42,42,REFUND_STATUS,REFUND_STATUS,0.601328,0.049719,0.852185,Intent is sufficiently confident and relevant historical evidence was retrieved.


In [119]:
pd.crosstab(
    auto_failures["human_intent"],
    auto_failures["predicted_intent"]
)

predicted_intent,ACCOUNT_ACCESS,CASHBACK,DELIVERY_DELAY,DELIVERY_DRIVER_ISSUE,DELIVERY_NOT_RECEIVED,DELIVERY_TRACKING,MISSING_ITEM,ORDER_CANCELLATION,ORDER_STATUS,PRODUCT_ISSUE,REFUND_STATUS,RETURN_REPLACEMENT,WRONG_ITEM_RECEIVED
human_intent,,,,,,,,,,,,,
ACCOUNT_ACCESS,0,1,1,1,0,0,0,0,0,0,0,0,0
ACCOUNT_SECURITY,1,0,0,0,0,0,1,0,0,0,0,0,0
CASHBACK,1,0,0,0,0,0,0,0,0,0,2,0,0
DELIVERY_DELAY,1,0,1,0,2,0,0,1,0,0,0,0,0
DELIVERY_DRIVER_ISSUE,0,0,0,1,0,0,0,0,0,0,0,0,0
DELIVERY_NOT_RECEIVED,0,0,0,0,0,0,0,0,0,1,0,0,1
DELIVERY_TRACKING,1,0,1,0,0,0,0,0,0,0,0,0,0
MISSING_ITEM,0,0,0,0,1,0,0,0,0,0,1,0,0
ORDER_CANCELLATION,0,0,0,0,0,1,0,0,0,0,0,0,0


In [120]:
SUPPORTED_INTENTS = [
    "DELIVERY_DELAY",
    "DELIVERY_NOT_RECEIVED",
    "DELIVERY_TRACKING",
    "DELIVERY_DRIVER_ISSUE",
    "ORDER_STATUS",
    "ORDER_CANCELLATION",
    "MISSING_ITEM",
    "WRONG_ITEM_RECEIVED",
    "RETURN_REPLACEMENT",
    "REFUND_STATUS",
    "ACCOUNT_ACCESS",
    "ACCOUNT_SECURITY",
    "UNEXPECTED_CHARGE",
    "CASHBACK",
    "PRODUCT_ISSUE",
    "OTHER"
]

In [121]:
def build_intent_prompt(query):

    intents = "\n".join(
        f"- {intent}" for intent in SUPPORTED_INTENTS
    )

    return f"""
You are an intent classifier for an Amazon customer support agent.

Classify the customer's message into exactly ONE of the following intents:

{intents}

INTENT DEFINITIONS:

DELIVERY_DELAY:
The promised or expected delivery date has passed.

DELIVERY_NOT_RECEIVED:
The customer has not received the package/order at all.

DELIVERY_TRACKING:
The customer is primarily asking about tracking, package status,
or where the package is, without a clearly established missed
delivery date.

DELIVERY_DRIVER_ISSUE:
Issue involving a delivery driver, carrier, delivery attempt,
delivery instructions, or address during delivery.

ORDER_STATUS:
General question about the state/status of an order.

ORDER_CANCELLATION:
Customer wants to cancel an order or reports an unexpected cancellation.

MISSING_ITEM:
The package arrived but one or more expected items are missing.

WRONG_ITEM_RECEIVED:
The customer received an incorrect product.

RETURN_REPLACEMENT:
The customer explicitly wants to return or replace an item.

REFUND_STATUS:
The customer is asking about a refund or missing/pending refund.

ACCOUNT_ACCESS:
Login, password, locked account, or inability to access the account.

ACCOUNT_SECURITY:
Hacked account, compromised account, unauthorized access, or suspicious
account activity.

UNEXPECTED_CHARGE:
Unrecognized, duplicate, or unexpected charge.

CASHBACK:
Expected cashback or reward has not been received.

PRODUCT_ISSUE:
Product is damaged, defective, broken, faulty, or not working.

OTHER:
The message does not provide enough evidence for a supported intent,
is generic/praise/unsupported, or does not fit the taxonomy.

IMPORTANT BOUNDARIES:
- A package delivered late is DELIVERY_DELAY.
- A package simply not received is DELIVERY_NOT_RECEIVED.
- Tracking/status uncertainty is DELIVERY_TRACKING.
- If an order arrived but an item is missing, use MISSING_ITEM.
- If the wrong product arrived, use WRONG_ITEM_RECEIVED.
- If a product is broken and the customer explicitly requests a return
  or replacement, use RETURN_REPLACEMENT.
- Prime is context, not an intent.
- If there is insufficient evidence to distinguish the intent,
  choose OTHER.

CUSTOMER MESSAGE:
{query}

Return ONLY the intent name.
"""

In [122]:
def classify_intent_groq(query):

    prompt = build_intent_prompt(query)

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [123]:
test_queries = [
    "My package was supposed to arrive yesterday but it still hasn't arrived.",
    "My order was delivered but one of the two items is missing.",
    "The tracking hasn't updated in three days.",
    "Someone hacked my Amazon account and changed my email.",
    "Way to drop the ball on customer service so pissed right now!"
]

for q in test_queries:
    print("\nCustomer:", q)
    print("Intent:", classify_intent_groq(q))


Customer: My package was supposed to arrive yesterday but it still hasn't arrived.
Intent: DELIVERY_DELAY

Customer: My order was delivered but one of the two items is missing.
Intent: MISSING_ITEM

Customer: The tracking hasn't updated in three days.
Intent: DELIVERY_TRACKING

Customer: Someone hacked my Amazon account and changed my email.
Intent: ACCOUNT_SECURITY

Customer: Way to drop the ball on customer service so pissed right now!
Intent: OTHER


In [125]:
import time

groq_predictions = []

for i, row in golden_df.iterrows():

    try:
        predicted = classify_intent_groq(row["customer_text"])

        groq_predictions.append({
            "golden_index": i,
            "human_intent": row["human_intent"],
            "groq_intent": predicted
        })

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(golden_df)}")

        # Small delay to be gentle with the free API
        time.sleep(0.2)

    except Exception as e:
        print(f"Error at row {i}: {e}")
        groq_predictions.append({
            "golden_index": i,
            "human_intent": row["human_intent"],
            "groq_intent": "API_ERROR"
        })

Processed 10/213
Processed 20/213
Processed 30/213
Processed 40/213
Processed 50/213
Processed 60/213
Processed 70/213
Processed 80/213
Processed 90/213
Processed 100/213
Processed 110/213
Processed 120/213
Processed 130/213
Processed 140/213
Processed 150/213
Processed 160/213
Processed 170/213
Processed 180/213
Processed 190/213
Processed 200/213
Processed 210/213


In [126]:
groq_eval = pd.DataFrame(groq_predictions)

print(groq_eval.shape)
groq_eval.head()

(213, 3)


,golden_index,human_intent,groq_intent
0,0,ACCOUNT_SECURITY,ACCOUNT_SECURITY
1,1,DELIVERY_NOT_RECEIVED,DELIVERY_NOT_RECEIVED
2,2,MISSING_ITEM,MISSING_ITEM
3,3,PRODUCT_ISSUE,RETURN_REPLACEMENT
4,4,REFUND_STATUS,REFUND_STATUS


In [127]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

valid = groq_eval[
    groq_eval["groq_intent"] != "API_ERROR"
]

accuracy = accuracy_score(
    valid["human_intent"],
    valid["groq_intent"]
)

macro_f1 = f1_score(
    valid["human_intent"],
    valid["groq_intent"],
    average="macro"
)

weighted_f1 = f1_score(
    valid["human_intent"],
    valid["groq_intent"],
    average="weighted"
)

print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

Accuracy:    0.6103
Macro F1:    0.6134
Weighted F1: 0.6133


In [128]:
print(
    classification_report(
        valid["human_intent"],
        valid["groq_intent"],
        zero_division=0
    )
)

                       precision    recall  f1-score   support

       ACCOUNT_ACCESS       0.77      0.67      0.71        15
     ACCOUNT_SECURITY       0.78      0.58      0.67        12
             CASHBACK       0.60      0.75      0.67         8
       DELIVERY_DELAY       0.27      0.55      0.36        11
DELIVERY_DRIVER_ISSUE       0.57      1.00      0.73         8
DELIVERY_NOT_RECEIVED       0.71      0.75      0.73        16
    DELIVERY_TRACKING       0.75      0.75      0.75        16
         MISSING_ITEM       0.80      0.67      0.73         6
   ORDER_CANCELLATION       0.86      0.63      0.73        19
         ORDER_STATUS       1.00      0.25      0.40        12
                OTHER       0.66      0.51      0.58        37
        PRODUCT_ISSUE       0.44      0.31      0.36        13
        REFUND_STATUS       0.62      0.76      0.68        17
   RETURN_REPLACEMENT       0.27      0.44      0.33         9
    UNEXPECTED_CHARGE       0.54      0.78      0.64  

In [129]:
groq_errors = groq_eval[
    groq_eval["human_intent"] != groq_eval["groq_intent"]
].copy()

print("Groq classification errors:", len(groq_errors))

pd.crosstab(
    groq_errors["human_intent"],
    groq_errors["groq_intent"]
)

Groq classification errors: 83


groq_intent,ACCOUNT_ACCESS,ACCOUNT_SECURITY,CASHBACK,DELIVERY_DELAY,DELIVERY_DRIVER_ISSUE,DELIVERY_NOT_RECEIVED,DELIVERY_TRACKING,MISSING_ITEM,ORDER_CANCELLATION,OTHER,PRODUCT_ISSUE,REFUND_STATUS,RETURN_REPLACEMENT,UNEXPECTED_CHARGE
human_intent,,,,,,,,,,,,,,
ACCOUNT_ACCESS,0,0,1,1,2,0,0,0,0,1,0,0,0,0
ACCOUNT_SECURITY,1,0,0,2,0,0,0,0,0,1,1,0,0,0
CASHBACK,0,0,0,0,0,0,0,0,0,1,0,1,0,0
DELIVERY_DELAY,0,0,1,0,0,1,0,0,1,0,1,0,0,1
DELIVERY_NOT_RECEIVED,0,1,0,1,0,0,0,0,0,0,1,0,1,0
DELIVERY_TRACKING,0,0,0,1,0,1,0,0,1,1,0,0,0,0
MISSING_ITEM,0,0,0,0,0,0,0,0,0,0,0,1,1,0
ORDER_CANCELLATION,0,0,0,1,0,1,1,0,0,2,0,0,1,1
ORDER_STATUS,0,0,1,1,0,1,1,0,0,4,0,1,0,0


In [130]:
groq_errors[
    [
        "golden_index",
        "human_intent",
        "groq_intent"
    ]
].head(30)

,golden_index,human_intent,groq_intent
3,3,PRODUCT_ISSUE,RETURN_REPLACEMENT
7,7,ORDER_CANCELLATION,OTHER
10,10,ORDER_STATUS,DELIVERY_NOT_RECEIVED
12,12,RETURN_REPLACEMENT,DELIVERY_TRACKING
24,24,OTHER,DELIVERY_DRIVER_ISSUE
27,27,ORDER_STATUS,OTHER
30,30,ORDER_STATUS,OTHER
40,40,ORDER_STATUS,DELIVERY_TRACKING
43,43,ACCOUNT_SECURITY,ACCOUNT_ACCESS
46,46,RETURN_REPLACEMENT,DELIVERY_DRIVER_ISSUE


In [131]:
decision_eval_v2 = decision_eval.merge(
    groq_eval[
        ["golden_index", "groq_intent"]
    ],
    on="golden_index",
    how="left"
)

print(decision_eval_v2.shape)

(213, 14)


In [132]:
decision_eval_v2["top_case_intent"] = None

for idx, row in golden_df.iterrows():

    results = retrieve_similar_cases(
        row["customer_text"],
        top_k=1
    )

    decision_eval_v2.loc[
        decision_eval_v2["golden_index"] == idx,
        "top_case_intent"
    ] = results.iloc[0]["predicted_intent"]

In [133]:
decision_eval_v2["groq_retrieval_agreement"] = (
    decision_eval_v2["groq_intent"]
    == decision_eval_v2["top_case_intent"]
)

In [134]:
decision_eval_v2["groq_retrieval_agreement"].value_counts()

groq_retrieval_agreement
True     112
False    101
Name: count, dtype: int64

In [135]:
RISK_INTENTS = {
    "ACCOUNT_SECURITY",
    "UNEXPECTED_CHARGE"
}

def make_decision_v2(
    groq_intent,
    top_similarity,
    retrieval_agreement
):
    # High-risk categories
    if groq_intent in RISK_INTENTS:
        return (
            "ESCALATE",
            f"{groq_intent} is treated as high-risk and requires human review."
        )

    # Unsupported / unclear category
    if groq_intent == "OTHER":
        return (
            "ESCALATE",
            "Message does not clearly fit a supported support intent."
        )

    # Weak historical evidence
    if top_similarity < 0.85:
        return (
            "ESCALATE",
            "Historical evidence is not sufficiently similar."
        )

    # LLM and retrieved evidence disagree
    if not retrieval_agreement:
        return (
            "ESCALATE",
            "The predicted intent conflicts with the most relevant historical case."
        )

    return (
        "AUTO_HANDLE",
        "Supported intent with strong historical evidence and consistent retrieval."
    )

In [136]:
decision_results = []

for _, row in decision_eval_v2.iterrows():

    decision, reason = make_decision_v2(
        groq_intent=row["groq_intent"],
        top_similarity=row["top_similarity"],
        retrieval_agreement=row["groq_retrieval_agreement"]
    )

    decision_results.append({
        "final_decision": decision,
        "final_reason": reason
    })

decision_results = pd.DataFrame(decision_results)

decision_eval_v2 = pd.concat(
    [
        decision_eval_v2.reset_index(drop=True),
        decision_results
    ],
    axis=1
)

In [137]:
print(
    decision_eval_v2["final_decision"].value_counts()
)

final_decision
ESCALATE       136
AUTO_HANDLE     77
Name: count, dtype: int64


In [139]:
auto = decision_eval_v2[
    decision_eval_v2["final_decision"] == "AUTO_HANDLE"
].copy()

auto_precision = (
    auto["groq_intent"] == auto["human_intent"]
).mean()

automation_rate = len(auto) / len(decision_eval_v2)

print(f"AUTO-HANDLE: {len(auto)}")
print(f"ESCALATE: {len(decision_eval_v2) - len(auto)}")
print(f"Automation rate: {automation_rate:.3f}")
print(f"Safe automation precision: {auto_precision:.3f}")

AUTO-HANDLE: 77
ESCALATE: 136
Automation rate: 0.362
Safe automation precision: 0.701


In [142]:
auto = decision_eval_v2[
    decision_eval_v2["final_decision"] == "AUTO_HANDLE"
].copy()

auto_precision = (
    auto["groq_intent"] == auto["human_intent"]
).mean()

automation_rate = len(auto) / len(decision_eval_v2)

print(f"AUTO-HANDLE: {len(auto)}")
print(f"ESCALATE: {len(decision_eval_v2) - len(auto)}")
print(f"Automation rate: {automation_rate:.3f}")
print(f"Safe automation precision: {auto_precision:.3f}")

AUTO-HANDLE: 77
ESCALATE: 136
Automation rate: 0.362
Safe automation precision: 0.701


In [145]:
print(
    auto_errors.groupby(
        ["human_intent", "groq_intent"]
    ).size().sort_values(ascending=False)
)

human_intent           groq_intent          
REFUND_STATUS          DELIVERY_DELAY           2
OTHER                  DELIVERY_DELAY           2
CASHBACK               REFUND_STATUS            1
ACCOUNT_ACCESS         DELIVERY_DRIVER_ISSUE    1
                       CASHBACK                 1
DELIVERY_DELAY         ORDER_CANCELLATION       1
                       CASHBACK                 1
DELIVERY_NOT_RECEIVED  PRODUCT_ISSUE            1
DELIVERY_TRACKING      DELIVERY_DELAY           1
ORDER_CANCELLATION     DELIVERY_NOT_RECEIVED    1
                       DELIVERY_TRACKING        1
DELIVERY_TRACKING      ORDER_CANCELLATION       1
MISSING_ITEM           REFUND_STATUS            1
ORDER_STATUS           DELIVERY_NOT_RECEIVED    1
                       CASHBACK                 1
OTHER                  ACCOUNT_ACCESS           1
ORDER_STATUS           REFUND_STATUS            1
OTHER                  DELIVERY_DRIVER_ISSUE    1
                       REFUND_STATUS            1
PRODU

In [146]:
print(
    auto_errors["groq_intent"].value_counts()
)

groq_intent
DELIVERY_DELAY           5
REFUND_STATUS            4
CASHBACK                 3
DELIVERY_NOT_RECEIVED    2
ORDER_CANCELLATION       2
DELIVERY_DRIVER_ISSUE    2
ACCOUNT_ACCESS           2
DELIVERY_TRACKING        1
MISSING_ITEM             1
PRODUCT_ISSUE            1
Name: count, dtype: int64


In [147]:
diagnostic_rows = []

for _, row in auto_errors.iterrows():

    golden_idx = row["golden_index"]

    query = golden_df.loc[
        golden_df.index == golden_idx,
        "customer_text"
    ].iloc[0]

    results = retrieve_similar_cases(
        query,
        top_k=1
    )

    top_case = results.iloc[0]

    diagnostic_rows.append({
        "customer_text": query,
        "human_intent": row["human_intent"],
        "groq_intent": row["groq_intent"],
        "similarity": top_case["similarity"],
        "historical_customer": top_case["customer_text"],
        "historical_response": top_case["support_text"]
    })

error_diagnostics = pd.DataFrame(diagnostic_rows)

pd.set_option("display.max_colwidth", 300)

display(error_diagnostics)

,customer_text,human_intent,groq_intent,similarity,historical_customer,historical_response
0,@115850 my order did not reached yet,ORDER_STATUS,DELIVERY_NOT_RECEIVED,0.902649,@115850 my order no 403-2881035-1260303 shown delivered but yet to reach me.,@495238 - so that we can do the needful. 2/3 ^RI
1,@115850 I didn't receive my promised cashback. Neither am I able to contact customer care. There's a robot on the other side. :( Help ASAP,DELIVERY_DELAY,CASHBACK,0.984067,@115850 I didn't receive my promised cashback. Neither am I able to contact customer care. There's a robot on the other side. :( Help ASAP,@192630 we'll reach out to you at the earliest. 2/2^EM
2,"Hey @115821, I'm waiting on that refund... thanks 😇",OTHER,REFUND_STATUS,0.957054,"Hey @115821, I'm waiting on that refund... thanks 😇",@175186 Hate to hear that you've not received your refund yet! Did you receive a confirmation about your refund? ^AD
3,Thanks @115821 for messing up my order just because I deleted an item that hadn't even shipped yet. Very disappointed.,DELIVERY_DELAY,ORDER_CANCELLATION,0.969377,Thanks @115821 for messing up my order just because I deleted an item that hadn't even shipped yet. Very disappointed.,"@446166 I'm sorry to hear about that, Jeanette. W/o personal or account info, could you tell us a little more about what happened? ^BL"
4,@AmazonHelp good condition. Just started refund process. Another item never arrived but got a refund on that one.,MISSING_ITEM,REFUND_STATUS,0.874527,@AmazonHelp Have started a refund. Just wanted some help with that. Have opted for Amazon pay by mistake. Please change that to Bank refund,@399506 Kindly reach our support team using the link provided earlier. We'll have it checked. ^GS
5,Hi @115830\nMy package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday? https://t.co/YGr1AdoFt3,DELIVERY_TRACKING,DELIVERY_DELAY,0.957139,Hi @115830\nMy package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday? https://t.co/YGr1AdoFt3,"@442368 My apologies for this. If you get in touch: https://t.co/JzP7hlA23B, we'll be happy to investigate. ^DC"
6,Another failed delivery from @117795 Getting fed up with this as have lost count of the number of deliveries that have failed,ACCOUNT_ACCESS,DELIVERY_DRIVER_ISSUE,0.952084,Another failed delivery from @117795 Getting fed up with this as have lost count of the number of deliveries that have failed,"@367863 Sorry to hear that Arley, is it the same carrier? ^RS"
7,"@115830 Now I can't get into my account at all, you're a bunch of thieving Ba****ds!",REFUND_STATUS,ACCOUNT_ACCESS,0.949399,"@115830 Now I can't get into my account at all, you're a bunch of thieving Ba****ds!",@283678 I am sorry to hear this. Can I ask have you received an email from our account specialist please? Please check your spam/junk folder as it may have been directed here.^GA
8,"Ummmm as an @115821 prime member, I’m highly disappointed that my shipping time has been extended😒😒",REFUND_STATUS,DELIVERY_DELAY,0.943847,"Ummmm as an @115821 prime member, I’m highly disappointed that my shipping time has been extended😒😒",@342693 I'm sorry about the delay! We aim to deliver by the date given in your order confirmation e-mail. Have we missed that date? ^TK
9,"@226254 @115821 Yep, I’ve definitely noticed more delayed packages in the last few months. Had one today that was supposed to arrive but didn’t.",REFUND_STATUS,DELIVERY_DELAY,0.957552,"@226254 @115821 Yep, I’ve definitely noticed more delayed packages in the last few months. Had one today that was supposed to arrive but didn’t.","@226618 I am sorry to hear this. Can I ask, what is the current status of the order on the tracking here please: https://t.co/Y5jpI9gRhE.^GA"


In [148]:
# Overall decision metrics
print("=== FINAL DECISION METRICS ===")

total = len(decision_eval_v2)
auto = decision_eval_v2[
    decision_eval_v2["final_decision"] == "AUTO_HANDLE"
]

correct_auto = auto[
    auto["groq_intent"] == auto["human_intent"]
]

print(f"Total cases: {total}")
print(f"Auto-handle: {len(auto)} ({len(auto)/total:.1%})")
print(f"Escalate: {total-len(auto)} ({(total-len(auto))/total:.1%})")
print(f"Safe automation precision: {len(correct_auto)/len(auto):.1%}")

=== FINAL DECISION METRICS ===
Total cases: 213
Auto-handle: 77 (36.2%)
Escalate: 136 (63.8%)
Safe automation precision: 70.1%


In [149]:
print("\n=== AUTO-HANDLE PRECISION BY INTENT ===")

auto["correct"] = (
    auto["groq_intent"] == auto["human_intent"]
)

print(
    auto.groupby("groq_intent")["correct"]
        .agg(["count", "sum", "mean"])
        .sort_values("count", ascending=False)
)


=== AUTO-HANDLE PRECISION BY INTENT ===
                       count  sum      mean
groq_intent                                
ORDER_CANCELLATION        14   12  0.857143
REFUND_STATUS             12    8  0.666667
ACCOUNT_ACCESS            10    8  0.800000
DELIVERY_DRIVER_ISSUE      8    6  0.750000
CASHBACK                   7    4  0.571429
DELIVERY_TRACKING          7    6  0.857143
DELIVERY_NOT_RECEIVED      7    5  0.714286
DELIVERY_DELAY             6    1  0.166667
PRODUCT_ISSUE              3    2  0.666667
MISSING_ITEM               2    1  0.500000
RETURN_REPLACEMENT         1    1  1.000000


In [150]:
HIGH_CONFIDENCE_INTENTS = {
    "ORDER_CANCELLATION",
    "DELIVERY_TRACKING",
    "ACCOUNT_ACCESS",
    "DELIVERY_DRIVER_ISSUE",
    "DELIVERY_NOT_RECEIVED",
    "RETURN_REPLACEMENT",
}

RISK_INTENTS = {
    "ACCOUNT_SECURITY",
    "UNEXPECTED_CHARGE",
}

LOW_CONFIDENCE_INTENTS = {
    "DELIVERY_DELAY",
    "CASHBACK",
    "MISSING_ITEM",
    "PRODUCT_ISSUE",
    "REFUND_STATUS",
    "ORDER_STATUS",
}

In [151]:
def make_decision_v4(
    groq_intent,
    top_similarity,
    retrieval_agreement
):

    # High-risk
    if groq_intent in RISK_INTENTS:
        return (
            "ESCALATE",
            f"{groq_intent} is high-risk and requires human review."
        )

    # Unsupported
    if groq_intent == "OTHER":
        return (
            "ESCALATE",
            "Message does not clearly fit a supported intent."
        )

    # Weak evidence
    if top_similarity < 0.85:
        return (
            "ESCALATE",
            "Historical evidence is insufficiently similar."
        )

    # Known difficult intents
    if groq_intent in LOW_CONFIDENCE_INTENTS:
        return (
            "ESCALATE",
            f"{groq_intent} has insufficient classification reliability "
            "for automatic handling."
        )

    # Require agreement for automatic handling
    if not retrieval_agreement:
        return (
            "ESCALATE",
            "Predicted intent conflicts with retrieved historical evidence."
        )

    return (
        "AUTO_HANDLE",
        "Supported high-confidence intent with strong historical evidence."
    )

In [152]:
auto_generation = []

for _, row in auto.iterrows():

    query = golden_df.loc[
        golden_df.index == row["golden_index"],
        "customer_text"
    ].iloc[0]

    results = retrieve_similar_cases(
        query,
        top_k=5
    )

    generated = generate_response(
        query,
        results
    )

    auto_generation.append({
        "golden_index": row["golden_index"],
        "customer_text": query,
        "human_intent": row["human_intent"],
        "groq_intent": row["groq_intent"],
        "top_similarity": row["top_similarity"],
        "generated_response": generated
    })

auto_generation_df = pd.DataFrame(auto_generation)

print(auto_generation_df.shape)
display(auto_generation_df.head())

(77, 6)


,golden_index,customer_text,human_intent,groq_intent,top_similarity,generated_response
0,1,"The @116090 app shows that my package was delivered yesterday, yet here I am without said package.",DELIVERY_NOT_RECEIVED,DELIVERY_NOT_RECEIVED,0.948658,"@116090 Sorry to hear your package shows as delivered but isn’t in your hands. Sometimes items are left in unexpected places (porch, side door, with a neighbor). Please try the quick‑check steps here: https://t.co/Q7Ftz6nj80. If the package still can’t be found, let us know and we’ll investigate..."
1,5,"@116875 tengo una cuenta bloqueada y no hay forma de entrar me aparece contraseña incorrecta, me podrian ayudar",ACCOUNT_ACCESS,ACCOUNT_ACCESS,0.929885,"Hola, ¿has recibido algún correo de Amazon con información sobre el bloqueo de tu cuenta y los pasos para desbloquearla? \n\nSi no lo tienes, intenta restablecer la contraseña usando la opción **“¿Olvidaste tu contraseña?”** en la página de inicio de sesión. También puedes buscar, justo debajo ..."
2,8,"@115823 I have not received cashback on amazon pay, where I can make complain ?",CASHBACK,CASHBACK,0.965109,"I’m sorry you haven’t received the cashback. To investigate, could you share the link to the specific Amazon Pay cashback offer you used? If you’d rather file a complaint directly, you can do so here: https://t.co/beaaDm0muc."
3,9,"@AmazonHelp Je ne peux pas me connecter , mon compte est verrouillé ! \nMy account is locked !!",ACCOUNT_ACCESS,ACCOUNT_ACCESS,0.860482,"Bonjour, nous sommes désolés que votre compte soit verrouillé. Sans communiquer d’informations personnelles, avez‑vous reçu un e‑mail de notre équipe de sécurité avec des instructions pour le déverrouiller ? Veuillez vérifier votre boîte de réception ainsi que les courriers indésirables et répon..."
4,10,@115850 my order did not reached yet,ORDER_STATUS,DELIVERY_NOT_RECEIVED,0.902649,"@115850 I’m sorry to hear your order hasn’t arrived yet. Please check the latest tracking information in Your Orders to see the current status. If the tracking shows it was delivered but you still haven’t received it, let us know so we can investigate further. Also, kindly avoid posting order nu..."


In [153]:
auto_generation_df.to_parquet(
    "../data/processed/auto_generated_responses.parquet",
    index=False
)

In [154]:
def build_judge_prompt(query, results, generated_response):

    evidence = ""

    for i, (_, r) in enumerate(results.iterrows(), 1):
        evidence += f"""
CASE {i}
Customer:
{r["customer_text"]}

Historical AmazonHelp response:
{r["support_text"]}
"""

    return f"""
You are evaluating an AI customer-support response.

Evaluate whether the generated response is appropriate for the customer
message and grounded in the historical AmazonHelp support behavior provided.

CUSTOMER MESSAGE:
{query}

HISTORICAL EVIDENCE:
{evidence}

GENERATED RESPONSE:
{generated_response}

Score each criterion from 1 to 5.

1. RELEVANCE
Does the response directly address the customer's issue?

2. GROUNDEDNESS
Is the response supported by the historical evidence?
Penalize claims that are not supported by the evidence.

3. NO_HALLUCINATION
Does the response avoid inventing policies, refunds, guarantees,
actions taken, links, or other unsupported facts?

4. CUSTOMER_APPROPRIATENESS
Is it concise, professional, empathetic, and useful?

5. RESOLUTION_ALIGNMENT
Does the response follow the general resolution approach demonstrated
by the historical AmazonHelp responses?

Return ONLY valid JSON in this exact format:

{{
  "relevance": <1-5>,
  "groundedness": <1-5>,
  "no_hallucination": <1-5>,
  "customer_appropriateness": <1-5>,
  "resolution_alignment": <1-5>,
  "overall": <1-5>,
  "reason": "<brief explanation>"
}}
"""

In [155]:
import json

def judge_response(query, results, generated_response):

    prompt = build_judge_prompt(
        query,
        results,
        generated_response
    )

    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content.strip()

    return json.loads(raw)

In [157]:
GENERATION_MODEL = "openai/gpt-oss-120b"
row = auto_generation_df.iloc[0]

results = retrieve_similar_cases(
    row["customer_text"],
    top_k=5
)

judge_result = judge_response(
    row["customer_text"],
    results,
    row["generated_response"]
)

judge_result

{'relevance': 5,
 'groundedness': 5,
 'no_hallucination': 5,
 'customer_appropriateness': 5,
 'resolution_alignment': 5,
 'overall': 5,
 'reason': 'The response directly addresses the missing‑package issue, mirrors the tone and structure of historical replies, uses the same support link and phone/chat suggestion found in prior cases, and does not introduce any unsupported claims.'}

In [ ]:
output_path = "../data/processed/llm_judge_results.parquet"

if os.path.exists(output_path):
    judge_results_df = pd.read_parquet(output_path)
    judge_results = judge_results_df.to_dict("records")
    completed_ids = set(judge_results_df["golden_index"])

    print(f"Resuming from {len(completed_ids)} completed cases")
else:
    judge_results = []
    completed_ids = set()

for _, row in auto_generation_df.iterrows():

    golden_index = row["golden_index"]

    if golden_index in completed_ids:
        continue

    while True:
        try:

            results = retrieve_similar_cases(
                row["customer_text"],
                top_k=5
            )

            judge = judge_response(
                row["customer_text"],
                results,
                row["generated_response"]
            )

            judge_results.append({
                "golden_index": golden_index,
                "relevance": judge["relevance"],
                "groundedness": judge["groundedness"],
                "no_hallucination": judge["no_hallucination"],
                "customer_appropriateness": judge["customer_appropriateness"],
                "resolution_alignment": judge["resolution_alignment"],
                "overall": judge["overall"],
                "reason": judge["reason"]
            })

            completed_ids.add(golden_index)

            pd.DataFrame(judge_results).to_parquet(
                output_path,
                index=False
            )

            print(
                f"Completed {len(completed_ids)}/"
                f"{len(auto_generation_df)}"
            )

            # Slow down between requests
            time.sleep(2)

            break

        except Exception as e:

            error = str(e)

            if "429" in error or "rate limit" in error.lower():

                print("Rate limit reached. Waiting 30 seconds...")
                time.sleep(30)

            else:
                print(f"Error on {golden_index}: {e}")
                print("Checkpoint saved. Stopping.")
                raise

Completed 1/77
Completed 2/77
Completed 3/77
Completed 4/77
Completed 5/77
Completed 6/77
Completed 7/77
Completed 8/77
Completed 9/77
Completed 10/77
Completed 11/77
Completed 12/77
Completed 13/77
Completed 14/77
Completed 15/77
Completed 16/77
Completed 17/77
Completed 18/77
Completed 19/77
Completed 20/77
Completed 21/77
Completed 22/77
Completed 23/77
Completed 24/77
Completed 25/77
Completed 26/77
Completed 27/77
Completed 28/77
Completed 29/77
Completed 30/77
Completed 31/77
Completed 32/77
Completed 33/77
Completed 34/77
Completed 35/77
Completed 36/77
Completed 37/77
Completed 38/77
Completed 39/77
Error on golden_index=108: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m28aaqc5erm8pzkzk668ynkq` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198811, Requested 1328. Please try again in 1m0.047999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing

In [159]:
judge_df = pd.read_parquet(
    "../data/processed/llm_judge_results.parquet"
)

print(judge_df.shape)
display(judge_df.head())

(39, 8)


,golden_index,relevance,groundedness,no_hallucination,customer_appropriateness,resolution_alignment,overall,reason
0,1,5,4,5,5,5,5,"The response directly addresses the missing package issue, follows the pattern of offering a link to troubleshooting steps and contact options seen in the historical examples, and does not introduce unsupported policies. The only slight deviation is the added generic advice about possible locati..."
1,5,5,4,5,5,5,5,"The response directly addresses the blocked account issue, mirrors the style of historical replies (asking about email, suggesting skip‑sign‑in and offering email support), does not introduce unsupported policies or links, and is concise, empathetic, and aligned with AmazonHelp's typical resolut..."
2,8,5,5,5,5,5,5,"The response directly addresses the cashback issue, mirrors historical patterns by requesting the offer link and providing the exact complaint form used previously, introduces no unsupported information, is concise and empathetic, and follows the established resolution workflow."
3,9,5,5,5,5,5,5,"The response directly addresses the locked‑account issue, mirrors the typical AmazonHelp approach of asking about a security email and suggesting the password‑reset flow, introduces no unsupported claims, and is concise, empathetic, and helpful."
4,10,5,5,5,5,5,5,"The response directly addresses the undelivered order, mirrors historical guidance (checking tracking, avoiding public order details), does not introduce unsupported policies or links, and is concise, empathetic, and aligned with past resolution patterns."


In [160]:
print("\n=== LLM JUDGE SUMMARY ===")

metrics = [
    "relevance",
    "groundedness",
    "no_hallucination",
    "customer_appropriateness",
    "resolution_alignment",
    "overall"
]

print(judge_df[metrics].describe().T)


=== LLM JUDGE SUMMARY ===
                          count      mean       std  min  25%  50%  75%  max
relevance                  39.0  4.948718  0.223456  4.0  5.0  5.0  5.0  5.0
groundedness               39.0  4.846154  0.365518  4.0  5.0  5.0  5.0  5.0
no_hallucination           39.0  4.974359  0.160128  4.0  5.0  5.0  5.0  5.0
customer_appropriateness   39.0  4.871795  0.338688  4.0  5.0  5.0  5.0  5.0
resolution_alignment       39.0  4.923077  0.269953  4.0  5.0  5.0  5.0  5.0
overall                    39.0  4.923077  0.269953  4.0  5.0  5.0  5.0  5.0


In [161]:
human_correct = (
    auto_generation_df["human_intent"]
    == auto_generation_df["groq_intent"]
)

In [162]:
judge_analysis = judge_df.merge(
    auto_generation_df[
        [
            "golden_index",
            "human_intent",
            "groq_intent"
        ]
    ],
    on="golden_index",
    how="left"
)

judge_analysis["intent_correct"] = (
    judge_analysis["human_intent"]
    == judge_analysis["groq_intent"]
)

print(
    judge_analysis.groupby("intent_correct")["overall"]
    .agg(["count", "mean"])
)

                count      mean
intent_correct                 
False               5  4.800000
True               34  4.941176


In [163]:
print(
    pd.crosstab(
        judge_analysis["intent_correct"],
        judge_analysis["overall"]
    )
)

overall         4   5
intent_correct       
False           1   4
True            2  32


In [164]:
# Select a fixed, reproducible 15-case human audit sample

audit_sample = (
    judge_df
    .sample(n=15, random_state=2026)
    .merge(
        auto_generation_df[
            [
                "golden_index",
                "customer_text",
                "human_intent",
                "groq_intent",
                "generated_response"
            ]
        ],
        on="golden_index",
        how="left"
    )
)

audit_sample = audit_sample[
    [
        "golden_index",
        "customer_text",
        "generated_response",
        "overall"
    ]
].reset_index(drop=True)

for i, row in audit_sample.iterrows():

    print("=" * 100)
    print(f"CASE {i + 1}")
    print("=" * 100)

    print("\nCUSTOMER:")
    print(row["customer_text"])

    print("\nAI RESPONSE:")
    print(row["generated_response"])

    print("\nLLM JUDGE OVERALL:")
    print(row["overall"])

    print()

CASE 1

CUSTOMER:
@115830 product delivered, box battered, gift inside also damaged. How to proceed? This is unacceptable.

AI RESPONSE:
Hi @115830, I’m sorry to hear your package arrived with a damaged box and a damaged gift inside—that’s certainly not the experience we want you to have. Could you let me know who the seller of the item is? You can find the seller name in your order details. Once I have that information, I can help you explore the appropriate next steps, such as a replacement or refund. In the meantime, you may review the available options here: https://t.co/9gFG9f02CQ.

LLM JUDGE OVERALL:
5

CASE 2

CUSTOMER:
@116316 #WheresMyStuff my package was "delivered"; but I haven't received it!! 😨

AI RESPONSE:
@116316 I'm sorry to hear your order shows as delivered but you haven’t received it. Could you please double‑check the latest tracking details and let us know what it says? Our support team can investigate further for you here: https://t.co/TK4yevTaND. Thank you for you

In [168]:
import re

def sanitize_response(response):
    response = re.sub(r"https?://\S+", "", response)
    response = re.sub(r"(?<!\w)@\w+", "", response)
    response = re.sub(r"\^?[A-Z]{1,4}\s*$", "", response)
    response = re.sub(r"__[^_]+__", "", response)

    response = re.sub(r"\s+", " ", response)
    response = re.sub(r"\s+([,.!?])", r"\1", response)

    return response.strip()

In [169]:
response = generate_response(
    conversation_context,
    results,
    decision
)

NameError: name 'conversation_context' is not defined